# **multimoda-rs**: Tutorial examples IVUS/OCT to centerline

> [!NOTE]
> The data needed to run this examples can be found on the official [multimoda-rs](https://https://github.com/yungselm/multimoda-rs) Github for download. Store the examples folder directly in root to run.

## Alining from files
The first example will focus on solving the problem of aligning frames within a pullback of either IVUS or OCT images.
We will start with gated IVUS images (systole/diastole) during two different states (e.g. rest/stress).
The .csv files are expected to set up in the following style:
```text
--------------------------------------------------------------------
|      185     |       5.32     |      2.37       |        0.0     |
|      185     |       5.12     |      2.46       |        0.0     |
|      ...     |       ...      |      ...        |        ...     |
```
where the first column is the frame index the point is from, the second to forth are x-, y- and z-coordinates. The naming conventions of the files are diastolic_contours.csv, diastolic_reference_points.csv, ... (see ./data). This is in alignment with the output of the [AIVUS-CAA software](https://https://github.com/AI-in-Cardiovascular-Medicine/AIVUS-CAA).

The first goal is to align the frames within a pullback by translating their centroids to a line and rotating them towards each other minimizing Hausdorff distance of the contours and catheter contours created from the image center. The influence of the catheter (which represents the image center) on the rotation can be adjusted by the number of points passed to catheter. If no catheter should be created just pass n_points=0.

In the same function states are aligned with each other (e.g. systole to diastole) and z-distance are averaged over the two states to have comparable frame positions. If heartrate is very different (e.g. rest to stress) a resampling is performed of the lower heartrate geometry.

Load packages multimodars, and for linking the numpy package. Check the print output that working directory was set correctly. Performance seems to be slower in the Jupyter notebook, consider running the examples in the console directly.

In [1]:
import os
from pathlib import Path
import numpy as np
import multimodars as mm

cwd = Path.cwd()
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "examples" / "data").exists():
        os.chdir(candidate / "examples" / "data")
        break
    elif (candidate / "data").exists():
        os.chdir(candidate / "data")
        break
print(f"Working directory: {os.getcwd()}")


Working directory: /mnt/d/00_coding/multimoda-rs/examples/data


In [2]:
# Install if needed
%pip install trimesh plotly scipy nbformat

# Imports
import trimesh
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def trimesh_to_mesh3d(mesh, color, name, opacity=0.6):
    """
    Convert a trimesh.Trimesh to a Plotly Mesh3d trace.
    """
    # get vertices and faces
    verts = mesh.vertices
    faces = mesh.faces
    return go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        color=color, opacity=opacity,
        name=name,
        flatshading=True
    )

def plot_pair(before_paths, after_paths, colors, titles):
    """
    before_paths, after_paths: list of two .obj file paths [dia, sys]
    colors: list of two colors (e.g. ['blue','red'])
    titles: [left_title, right_title]
    """
    before_meshes = [trimesh.load(p) for p in before_paths]
    after_meshes  = [trimesh.load(p) for p in after_paths]

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type":"scene"}, {"type":"scene"}]],
        subplot_titles=titles
    )

    for mesh, color, label in zip(before_meshes, colors, ["diastole","systole"]):
        fig.add_trace(
            trimesh_to_mesh3d(mesh, color, f"before_{label}"),
            row=1, col=1
        )
    for mesh, color, label in zip(after_meshes, colors, ["diastole","systole"]):
        fig.add_trace(
            trimesh_to_mesh3d(mesh, color, f"after_{label}"),
            row=1, col=2
        )

    # link camera on both scenes
    camera = dict(
        eye=dict(x=1.5, y=1.5, z=1.0)
    )
    fig.update_layout(
        width=900, height=450,
        # apply same camera to both
        scene_camera=camera,
        scene2_camera=camera,
        # enforce equal scaling on x/y/z for both subplots
        scene=dict(
            aspectmode="data"
        ),
        scene2=dict(
            aspectmode="data"
        ),
        margin=dict(l=0, r=0, t=30, b=0)
    )
    fig.show()

# Paths “before” geometries
before_paths = [
    "output/unprocessed/dia_lumen.obj",
    "output/unprocessed/sys_lumen.obj",
]

# Paths “after” (processed) meshes
after_paths = [
    "output/rest/lumen_000_full.obj",    # diastole post
    "output/rest/lumen_029_full.obj",    # systole post
]

colors = ["royalblue", "firebrick"]

titles = ["Before Processing", "After Processing"]

plot_pair(before_paths, after_paths, colors, titles)


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


ValueError: string is not a file: `output/unprocessed/dia_lumen.obj`

The data is now neatly ordered in pairs (e.g. diastolic and systolic geometry). Every geometry has contours for lumen and walls and a created catheter. The reference point will be used to align the geometry to the centerline. All points corresponding to a contour are also save in a contour struct.

In [ ]:
print(f"Example of PyGeometryPair:\n{rest}")
print(f"Example of PyGeometry:\n{rest.geom_a}")
print(f"Example of PyFrame:\n{rest.geom_a.frames[0]}")
print(f"Example of PyContour:\n{rest.geom_a.frames[0].lumen}")
print(f"Example of Extras:\n{rest.geom_a.frames[0].extras}")
print(f"Example of PyContourPoint:\n{rest.geom_a.frames[0].lumen.points[0]}")


Additionally is it possible to get different measurements, regarding stenosis, directly from the objects now:

In [ ]:
# Summary over PyGeometryPair
print(f"Summary over PyGeometryPair object (dia_geom: (mla [mm^2], max. stenosis, stenosis length [mm]), sys_geom...):\n{rest.get_summary()[0]}\n \
      table (contour id, diastolic area, diastolic elliptic ratio, systolic area, systolic elliptic ratio, z coordinates):\n{np.array(rest.get_summary()[1])}")
# Summary over PyGeometry
print(f"PyGeometry (mla [mm^2], max. stenosis, stenosis length [mm]):\n {rest.geom_a.get_summary()[0]}")
# or more specific per contour
print(rest.geom_a.frames[0].lumen.get_area())
print(rest.geom_a.frames[-1].lumen.get_elliptic_ratio())


The four pairs represent all 4 possible comparison in gated images, as for example in coronary artery anomalies (rest pulsatile lumen deformation, stress pulsatile lumen deformation, stress-induced diastolic lumen deformation and stress-induced systolic lumen deformation). See also paper:

<img src="../paper/figures/Figure1.jpg" alt="States figure" width="500"/>

The idea behind the interpolation steps, is to provide a set of geometries that can be used to render a video of the compression over time (e.g. in blender, see the example script). Blender allows for an easy open sourec solution. Just navigate to the directory, and run blender with context:
```bash
cd '.\Program Files\Blender Foundation\Blender 4.4\'
.\blender.exe -con 
```
Then just copy/paste the script in the `Scripting` tab and run it. The result will look something like this:
<img src="../docs/figures/animation_stress_induced_systolic_deformation.gif" alt="Deformation Gif" width="500"/>

Scripts can be adjusted to individual need. If no animations are planned interpolation steps can just be set to 0.

## Coronary Artery Disease Case
So far we only did dynamic comparison in coronary artery disease. However `multimodars`can also be used to reconstruct single cases with coronary artery
disease and then reconstruct the different layers (catheter, lumen, eem and wall)
to create a 3D model of the stenosis.

In [ ]:
cad, _ = mm.from_file_single(
    input_path="ivus_full",
    diastole=True,
    step_rotation_deg=0.1, 
    range_rotation_deg=90, 
    image_center=(4.5, 4.5),
    radius=0.5,
    n_points=20,
    write_obj=True,
    watertight=False, # creates shell
    contour_types=[mm.PyContourType.Lumen, mm.PyContourType.Eem, mm.PyContourType.Catheter, mm.PyContourType.Wall],
    output_path="output/cad",
)
cad_aligned = cad.center_to_contour(mm.PyContourType.Eem)
mm.to_obj(cad_aligned, "output/cad", watertight=False, contour_types=[mm.PyContourType.Lumen, mm.PyContourType.Eem, mm.PyContourType.Catheter, mm.PyContourType.Wall], filename_prefix="aligned")

# load the two meshes
lumen = trimesh.load("output/cad/lumen_cad.obj")
eem = trimesh.load("output/cad/eem_cad.obj")
wall = trimesh.load("output/cad/wall_cad.obj")
lumen_aligned = trimesh.load("output/cad/aligned_lumen.obj")
eem_aligned = trimesh.load("output/cad/aligned_eem.obj")
wall_aligned = trimesh.load("output/cad/aligned_wall.obj")

# create traces
trace_lumen = trimesh_to_mesh3d(lumen, 'firebrick', 'Lumen', 1.0)
trace_eem = trimesh_to_mesh3d(eem, 'royalblue', 'External Elastic Membrane', 0.6)
trace_wall = trimesh_to_mesh3d(wall, 'white', 'Coronary Wall', 0.5)
trace_lumen_aligned = trimesh_to_mesh3d(lumen_aligned, 'firebrick', 'Lumen', 1.0)
trace_eem_aligned = trimesh_to_mesh3d(eem_aligned, 'royalblue', 'External Elastic Membrane', 0.6)
trace_wall_aligned = trimesh_to_mesh3d(wall_aligned, 'white', 'Coronary Wall', 0.5)

# Create a subplot layout with 2 scenes side-by-side
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=("CAD (Lumen aligned)", "CAD (EEM aligned)")
)

# Add traces to respective subplots
fig.add_trace(trace_lumen, row=1, col=1)
fig.add_trace(trace_eem, row=1, col=1)
fig.add_trace(trace_wall, row=1, col=1)

fig.add_trace(trace_lumen_aligned, row=1, col=2)
fig.add_trace(trace_eem_aligned, row=1, col=2)
fig.add_trace(trace_wall_aligned, row=1, col=2)

# Set up camera
camera = dict(eye=dict(x=1.5, y=1.5, z=1.0))

# Layout updates
fig.update_layout(
    # title_text="CAD Mesh Comparison: Original vs. Aligned",
    width=900, height=450,
    scene=dict(
        camera=camera,
        aspectmode='data',
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z"
    ),
    scene2=dict(
        camera=camera,
        aspectmode='data',
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z"
    ),
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

## Stent example
This can also be used for pre-and post-stenting comparison (here example of stenting an intramural course of a coronary artery anomaly):

In [ ]:
_, _, dia, sys, _ = mm.from_file_full(
    input_path_ab="ivus_prestent",
    input_path_cd="ivus_poststent",
    step_rotation_deg=0.1,
    range_rotation_deg=45,
    watertight=False,
    output_path_ab="output/stent_rest",
    output_path_cd="output/stent_stress",
    output_path_ac="output/stent_diastole",
    output_path_bd="output/stent_systole",
    interpolation_steps=0,
    )

# load the two meshes
mesh_dia = trimesh.load("output/stent_diastole/lumen_000_stent.obj")
mesh_sys = trimesh.load("output/stent_diastole/lumen_001_stent.obj")

# create traces
trace_dia = trimesh_to_mesh3d(mesh_dia, 'royalblue', 'Before (mesh_000)')
trace_sys = trimesh_to_mesh3d(mesh_sys, 'firebrick', 'After (mesh_001)')

# define a canonical camera position
camera = dict(eye=dict(x=1.5, y=1.5, z=1.0))

# build and show figure
fig = go.Figure(data=[trace_dia, trace_sys])
fig.update_layout(
    title="Post-processing: Prestenting vs Poststenting",
    width=600, height=600,
    scene=dict(
        aspectmode="data",    # equal scales on x/y/z
        camera=camera,
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    margin=dict(l=0, r=0, t=30, b=0)
)
fig.show()


However, for these pre- and poststenting comparisons the original from_file approach is computationally more expensive, than using the more flexible .from_array() approach:

In [ ]:
before_arr = np.genfromtxt("ivus_prestent/diastolic_contours.csv", delimiter='\t')
before_ref = np.genfromtxt("ivus_prestent/diastolic_reference_points.csv", delimiter='\t')
after_arr = np.genfromtxt("ivus_poststent/diastolic_contours.csv", delimiter='\t')
after_ref = np.genfromtxt("ivus_poststent/diastolic_reference_points.csv", delimiter='\t')

before_input_data = mm.numpy_to_inputdata(
    lumen_arr=before_arr,
    ref_point=before_ref,
    record=None,
    diastole=True,
    label="prestent",
)

after_input_data = mm.numpy_to_inputdata(
    lumen_arr=after_arr,
    ref_point=after_ref,
    record=None,
    diastole=True,
    label="poststent",
)

pair, _ = mm.from_array_singlepair(
    input_data_a=before_input_data,
    input_data_b=after_input_data,
    step_rotation_deg=0.01,
    range_rotation_deg=30,
    output_path="output/stent_comparison",
)

Or you can reconstruct a a single 3D geometry from OCT for example.

In [ ]:
oct_raw = np.genfromtxt("oct_single/oct_contours_raw.csv", delimiter=',')
oct_ref = np.genfromtxt("oct_single/oct_ref.csv", delimiter=',')

oct_input_data = mm.numpy_to_inputdata(
    lumen_arr=oct_raw,
    ref_point=oct_ref,
    record=None,
    diastole=True,
    label="oct",
)

oct_recon, _ = mm.from_array_single(
    input_data=oct_input_data,
    step_rotation_deg=0.01,
    range_rotation_deg=6,
    image_center=(5.0, 5.0),
    radius=0.5,
    n_points=40,
    write_obj=False,
    watertight=False,
    output_path="data/output/oct",
    smooth=False,
)

frame = oct_recon.get_frame_at_z(34.8)
id = frame.id + 1
oct_replaced = oct_recon.replace_frame(id, frame)

# Write the OCT geometry to disk and visualise it
mm.to_obj(
    oct_recon,
    "output/oct",
    watertight=False,
    contour_types=[mm.PyContourType.Lumen],
    filename_prefix="oct",
)

mm.to_obj(
    oct_replaced,
    "output/oct",
    watertight=False,
    contour_types=[mm.PyContourType.Lumen],
    filename_prefix="oct_replaced",
)

oct_mesh = trimesh.load("output/oct/oct_lumen.obj")
oct_mesh_fixed = trimesh.load("output/oct/oct_replaced_lumen.obj")

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=["OCT Lumen", "OCT Replaced"],
)

for trace in [trimesh_to_mesh3d(oct_mesh, 'royalblue', 'OCT Lumen')]:
    fig.add_trace(trace, row=1, col=1)

for trace in [trimesh_to_mesh3d(oct_mesh_fixed, 'tomato', 'OCT Replaced')]:
    fig.add_trace(trace, row=1, col=2)

scene_cfg = dict(aspectmode='data', camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)))
fig.update_layout(
    title='OCT single geometry - Original vs Replaced',
    width=1200, height=600,
    scene=scene_cfg,
    scene2=scene_cfg,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## Alignment from array
While the alignment from file is one option, the more flexible option is to create Geometries directly from numpy array, and then perform the same operations with these Geometries. It is enough to provide contour coordinates and a reference point for the different states that should be compared.

In [ ]:
record = np.genfromtxt("ivus_rest/combined_sorted_manual.csv", delimiter=',', skip_header=1)

dia_cont = np.genfromtxt("ivus_rest/diastolic_contours.csv", delimiter='\t')
dia_ref = np.genfromtxt("ivus_rest/diastolic_reference_points.csv", delimiter='\t')

sys_cont = np.genfromtxt("ivus_rest/systolic_contours.csv", delimiter='\t')
sys_ref = np.genfromtxt("ivus_rest/systolic_reference_points.csv", delimiter='\t')

rest_dia_input_data = mm.numpy_to_inputdata(
    lumen_arr=dia_cont,
    ref_point=dia_ref,
    record=record,
    diastole=True,
    label="diastole_rest",
)

rest_sys_input_data = mm.numpy_to_inputdata(
    lumen_arr=sys_cont,
    ref_point=sys_ref,
    record=record,
    diastole=False,
    label="systole_rest",
)

# Actual function call
rest, (dia_logs, sys_logs) = mm.from_array_singlepair(
    input_data_a=rest_dia_input_data,
    input_data_b=rest_sys_input_data,
    step_rotation_deg=0.01,
    range_rotation_deg=60,
    output_path="output/rest_array",
    interpolation_steps=0,
    smooth=True,
    postprocessing=True,
)


## Align with centerline
After aligning the frames within a geometry the alignment with a CCTA centerline can be performed by providing three different points. Here the example of a anomalous coronary artery where a point for the aorta one for the proximal part of the vessel and one for the distal part are provided.

<img src="../docs/figures/Alignment3p.png" alt="Alignment figure" width="500"/>

In [ ]:
rest, (dia_logs, sys_logs) = mm.from_file_singlepair(
    input_path="ivus_rest",
    labels=["aligned_dia", "aligned_sys", "", ""],
    output_path="output/rest",
)

cl_raw = np.genfromtxt("centerline_raw.csv", delimiter=',')
cl_converted = mm.numpy_to_centerline(cl_raw)

print(cl_raw)

aligned_geometry, resampled_cl = mm.align_three_point(
    centerline=cl_converted,
    geometry_pair=rest,
    aortic_ref_pt=(12.2605, -201.3643, 1751.0554),
    upper_ref_pt=(11.7567, -202.1920, 1754.7975),
    lower_ref_pt=(15.6605, -202.1920, 1749.9655),
    write=True,
    watertight=False,
    interpolation_steps=0,
)

print(resampled_cl)

In [ ]:
# Paths “before” geometries
before_paths = [
    "output/rest/lumen_000_full.obj",
    "output/rest/lumen_029_full.obj",
]

# Paths “after” (processed) meshes
after_paths = [
    "output/aligned/lumen_000_None.obj",    # diastole post
    "output/aligned/lumen_001_None.obj",    # systole post
]

colors = ["royalblue", "firebrick"]

titles = ["Before Alignment", "After Alignment"]

plot_pair(before_paths, after_paths, colors, titles)